### Hỗ trợ:
 - Github copilot dùng để trợ giúp viết báo cáo bằng latex nhanh hơn, nhưng cần kiểm tra kỹ vì có thể nó sẽ viết sai hoặc không đúng ý mình.
 - Giúp viết các hàm kiểm chứng kết quả nhanh hơn và giải thích các kết quả trả về của các hàm có sẵn trong NumPy, SciPy, SymPy. 

## Lưu ý: 
* Toàn bộ các kết quả đều được so sánh bằng số thực với sai số cho phép để tránh lệch do làm tròn.
* Các hàm chính như gaussian_eliminate, back_substitution, inverse, determinant, rank_and_basis đều được viết để xử lý các ma trận số thực (float) với EPS nhằm đảm bảo tốc độ.
* Các hàm phụ như dinh_dang_hien_thi, in_ma_tran_dep được dùng để chuyển đổi và hiển thị ma trận một cách dễ đọc.
* các hàm print() đều chạy hàm chính trong đó. nên sử dụng hàm print để hiện thị kết quả thay vì gọi trực tiếp hàm chính để đảm bảo kết quả được hiển thị đúng định dạng, so sánh và dễ đọc.

In [14]:
%pip install numpy -q
import numpy as np
from gaussian import *
from determinant import determinant, print_determinant
from inverse import inverse, print_inverse
from rank_basis import rank_and_basis, print_rank_and_basis


def verify_solution(A, x, b):
	"""Kiểm chứng x có phải nghiệm của Ax=b không (chỉ dùng cho nghiệm số cụ thể)."""
	A_np = np.array(A, dtype=float)
	x_np = np.array([float(val) for val in x], dtype=float)
	b_np = np.array([float(val) for val in b], dtype=float)
	Ax_np = A_np @ x_np
	return np.allclose(Ax_np, b_np)


def compare(check):
	if check:
		print("Đáp án đúng")
	else:
		print("Đáp án sai")

Note: you may need to restart the kernel to use updated packages.


## Test: Gaussian Elimination

In [15]:
# TEST 1: Hệ có nghiệm duy nhất
print("TEST 1: Hệ có nghiệm duy nhất")
A1 = [[1, 2, 0, 2], [3, 5, -1, 6], [2, 4, 1, 2], [2, 0, -7, 11]]
b1 = [6, 17, 12, 7]
x1 = gaussian_eliminate(A1, b1)[1]
print_gaussian_eliminate(A1, b1)
print("Kiểm chứng:")
if x1 and not isinstance(x1[0], str):
	check = verify_solution(A1, x1, b1)
	compare(check)
print()

# TEST 2: Hệ vô nghiệm
print("TEST 2: Hệ vô nghiệm")
A2 = [[2, -4, -1], [1, -3, 1], [3, -5, -3]]
b2 = [1, 1, 2]
print_gaussian_eliminate(A2, b2)

# TEST 3: Hệ có vô số nghiệm
print("TEST 3: Hệ có vô số nghiệm")
A3 = [[1, -2, -1], [2, -3, 1], [3, -5, 0], [1, 0, 5]]
b3 = [1, 6, 7, 9]
print_gaussian_eliminate(A3, b3)

# TEST 4: Trường hợp đặc biệt - Hệ 1 phương trình 1 ẩn
print("TEST 4: Trường hợp đặc biệt - Hệ 1 phương trình 1 ẩn")
A4 = [[5]]
b4 = [10]
print_gaussian_eliminate(A4, b4)

# TEST 5: Trường hợp đặc biệt - Hệ 2 phương trình 4 ẩn
print("TEST 5: Trường hợp đặc biệt - Hệ 2 phương trình 4 ẩn")
A5 = [[1, 1, 1, 1], [2, 1, 3, 2]]
b5 = [4, 7]
print_gaussian_eliminate(A5, b5)

# TEST 6: Partial pivoting (pivot đầu = 0, cần đổi hàng)
print("TEST 6: Partial pivoting")
A6 = [[0, 2, 3], [1, -1, 2], [4, 1, -2]]
b6 = [5, 3, 1]
print_gaussian_eliminate(A6, b6)

TEST 1: Hệ có nghiệm duy nhất
Hệ có nghiệm duy nhất.
Ma trận sau khi đã được khử:
[          3           5          -1           6 ]
[          0    -3.33333    -6.33333           7 ]
[          0           0         0.4        -0.6 ]
[          0           0           0        0.25 ]
Nghiệm hệ: ['2', '3', '-2', '-1']
Số lần hoán đổi hàng: 2

Kiểm chứng:
Đáp án đúng

TEST 2: Hệ vô nghiệm
Hệ vô nghiệm.
Ma trận sau khi đã được khử:
[          3          -5          -3 ]
[          0    -1.33333           2 ]
[          0           0           0 ]
Nghiệm hệ: vô nghiệm
Số lần hoán đổi hàng: 1

TEST 3: Hệ có vô số nghiệm
Hệ có vô số nghiệm.
Ma trận sau khi đã được khử:
[          3          -5           0 ]
[          0     1.66667           5 ]
[          0           0           0 ]
[          0           0           0 ]
Nghiệm hệ: ['9 - 5*t', '4 - 3*t', 't']
Số lần hoán đổi hàng: 2

TEST 4: Trường hợp đặc biệt - Hệ 1 phương trình 1 ẩn
Hệ có nghiệm duy nhất.
Ma trận sau khi đã được khử:
[ 

## Test: Back Substitution

In [16]:
# TEST 1: Ma trận tam giác trên - Hệ có nghiệm duy nhất
print("TEST 1: Ma trận tam giác trên - Hệ có nghiệm duy nhất")
U1 = [[2, 1, 3], [0, 4, 2], [0, 0, 1]]
c1 = [13, 18, 5]
print_back_substitution(U1, c1)
x = back_substitution(U1, c1)
print("Kiểm chứng:")
if x and not isinstance(x[0], str):
	flag = verify_solution(U1, x, c1)
	compare(flag)
print()

# TEST 2: Ma trận tam giác trên - Hệ vô nghiệm
print("TEST 2: Ma trận tam giác trên - Hệ vô nghiệm")
U2 = [[1, 2], [0, 0]]
c2 = [5, 1]
print_back_substitution(U2, c2)

# TEST 3: Ma trận tam giác trên - Hệ vô số nghiệm
print("TEST 3: Ma trận tam giác trên - Hệ vô số nghiệm")
U3 = [[1, 1, 2], [0, 0, 0], [0, 0, 0]]
c3 = [5, 0, 0]
print_back_substitution(U3, c3)

# TEST 4: Trường hợp đặc biệt - Ma trận 1x1
print("TEST 4: Trường hợp đặc biệt - Ma trận 1x1")
U4 = [[3]]
c4 = [9]
print_back_substitution(U4, c4)

# TEST 5: Dữ liệu không hợp lệ (U không vuông)
print("TEST 5: Dữ liệu không hợp lệ (U không vuông)")
U5 = [[1, 2, 3, 4], [0, 0, 1, 2]]
c5 = [10, 5]
try:
	print_back_substitution(U5, c5)
except ValueError as e:
	print("Đã bắt lỗi đúng:", e)

TEST 1: Ma trận tam giác trên - Hệ có nghiệm duy nhất
Ma trận U:
[          2           1           3 ]
[          0           4           2 ]
[          0           0           1 ]
Vector c: ['13', '18', '5']
Nghiệm hệ: ['-2', '2', '5']

Kiểm chứng:
Đáp án đúng

TEST 2: Ma trận tam giác trên - Hệ vô nghiệm
Ma trận U:
[          1           2 ]
[          0           0 ]
Vector c: ['5', '1']
Nghiệm hệ: vô nghiệm

TEST 3: Ma trận tam giác trên - Hệ vô số nghiệm
Ma trận U:
[          1           1           2 ]
[          0           0           0 ]
[          0           0           0 ]
Vector c: ['5', '0', '0']
Nghiệm hệ: ['5 - t1 - 2*t2', 't1', 't2']

TEST 4: Trường hợp đặc biệt - Ma trận 1x1
Ma trận U:
[          3 ]
Vector c: ['9']
Nghiệm hệ: ['3']

TEST 5: Dữ liệu không hợp lệ (U không vuông)
Đã bắt lỗi đúng: U phải là ma trận vuông (n x n).


## Test: Determinant

In [17]:
# TEST 1: Ma trận 2x2
print("TEST 1: Ma trận 2x2")
A1 = [[2, 3], [1, 5]]
print_determinant(A1)

# TEST 2: Ma trận 3x3 
print("TEST 2: Ma trận 3x3")
A2 = [[1, 2, 3], [0, 1, 4], [5, 6, 0]]
print_determinant(A2)

# TEST 3: Ma trận có các hàng phụ thuộc tuyến tính (hàng 2 = 2*hàng 1) - Định thức = 0
print("TEST 3: Ma trận phụ thuộc tuyến tính")
A3 = [[1, 2, 3], [2, 4, 6], [0, 1, 2]]
print_determinant(A3)

# TEST 4: Ma trận 4x4
print("TEST 4: Ma trận 4x4")
A4 = [[1, 0, 2, -1], [3, 0, 0, 5], [2, 1, 4, -3], [1, 0, 5, 0]]
print_determinant(A4)

# TEST 5: Ma trận đơn vị - Định thức = 1
print("TEST 5: Ma trận đơn vị")
A5 = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
print_determinant(A5)

# TEST 6: Ma trận đường chéo - Định thức = tích các phần tử đường chéo
print("TEST 6: Ma trận đường chéo")
A6 = [[2, 0, 0, 0], [0, 3, 0, 0], [0, 0, -1, 0], [0, 0, 0, 4]]
print_determinant(A6)

TEST 1: Ma trận 2x2
Ma trận: 
[          2           3 ]
[          1           5 ]
Định thức: 7

TEST 2: Ma trận 3x3
Ma trận: 
[          1           2           3 ]
[          0           1           4 ]
[          5           6           0 ]
Định thức: 1

TEST 3: Ma trận phụ thuộc tuyến tính
Ma trận: 
[          1           2           3 ]
[          2           4           6 ]
[          0           1           2 ]
Định thức: 0

TEST 4: Ma trận 4x4
Ma trận: 
[          1           0           2          -1 ]
[          3           0           0           5 ]
[          2           1           4          -3 ]
[          1           0           5           0 ]
Định thức: 30

TEST 5: Ma trận đơn vị
Ma trận: 
[          1           0           0 ]
[          0           1           0 ]
[          0           0           1 ]
Định thức: 1

TEST 6: Ma trận đường chéo
Ma trận: 
[          2           0           0           0 ]
[          0           3           0           0 ]
[          

## Test: Matrix Inverse

In [18]:
# TEST 1: Ma trận 2x2
A1 = [[2, 3], [1, 5]]
print("TEST 1: Ma trận 2x2")
print_inverse(A1)

# TEST 2: Ma trận 3x3
A2 = [[1, 2, 3], [0, 1, 4], [5, 6, 0]]
print("TEST 2: Ma trận 3x3")
print_inverse(A2)

# TEST 3: Ma trận đơn vị (Identity) - nghịch đảo của I chính là I
A3 = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
print("TEST 3: Ma trận đơn vị")
print_inverse(A3)

# TEST 4: Ma trận không khả nghịch (singular)
A4 = [[1, 2, 3], [2, 4, 6], [0, 1, 2]]
print("TEST 4: Ma trận không khả nghịch det = 0")
print_inverse(A4)

# TEST 5: Ma trận 4x4
A5 = [[1, 0, 2, -1], [3, 0, 0, 5], [2, 1, 4, -3], [1, 0, 5, 0]]
print("TEST 5: Ma trận 4x4")
print_inverse(A5)

TEST 1: Ma trận 2x2
Ma trận A:
[          2           3 ]
[          1           5 ]

Ma trận nghịch đảo A^-1:
[   0.714286   -0.428571 ]
[  -0.142857    0.285714 ]

TEST 2: Ma trận 3x3
Ma trận A:
[          1           2           3 ]
[          0           1           4 ]
[          5           6           0 ]

Ma trận nghịch đảo A^-1:
[        -24          18           5 ]
[         20         -15          -4 ]
[         -5           4           1 ]

TEST 3: Ma trận đơn vị
Ma trận A:
[          1           0           0 ]
[          0           1           0 ]
[          0           0           1 ]

Ma trận nghịch đảo A^-1:
[          1           0           0 ]
[          0           1           0 ]
[          0           0           1 ]

TEST 4: Ma trận không khả nghịch det = 0
Ma trận A:
[          1           2           3 ]
[          2           4           6 ]
[          0           1           2 ]

Ma trận A không khả nghịch do det(A) = 0.

TEST 5: Ma trận 4x4
Ma trận A:
[  

## Test: Rank and Basis Calculation

In [19]:
# Test 1: rank đầy đủ, null space rỗng
A1 = [[1, 2], [3, 4]]
print("TEST 1: Ma trận vuông khả nghịch")
print_rank_and_basis(A1)

# Test 2: rank thiếu, có null space
A2 = [[1, 2, 3], [2, 4, 6], [0, 1, 2]]
print("TEST 2: Ma trận phụ thuộc tuyến tính")
print_rank_and_basis(A2)

# Test 3: ma trận chữ nhật m < n
A3 = [[1, 0, 2, -1], [2, 1, 4, -3]]
print("TEST 3: Ma trận chữ nhật")
print_rank_and_basis(A3)

# Test 4: Ma trận rank = 1 (tất cả hàng là bội số của nhau)
print("\nTEST 4: Ma trận rank = 1")
A4 = [[1, 2, 3], [2, 4, 6], [3, 6, 9]]
print_rank_and_basis(A4)

# Test 5: Ma trận zero - tất cả phần tử = 0
print("\nTEST 5: Ma trận zero (rank = 0)")
A5 = [[0, 0, 0], [0, 0, 0]]
print_rank_and_basis(A5)

TEST 1: Ma trận vuông khả nghịch
Ma trận:
[          1           2 ]
[          3           4 ]
Hạng (rank): 2

Cơ sở không gian cột:
  [1 3]
  [2 4]

Cơ sở không gian dòng:
  [3 4]
  [0 0.666667]

Cơ sở không gian nghiệm:
  Không gian nghiệm chỉ chứa vector 0.

TEST 2: Ma trận phụ thuộc tuyến tính
Ma trận:
[          1           2           3 ]
[          2           4           6 ]
[          0           1           2 ]
Hạng (rank): 2

Cơ sở không gian cột:
  [1 2 0]
  [2 4 1]

Cơ sở không gian dòng:
  [2 4 6]
  [0 1 2]

Cơ sở không gian nghiệm:
  [1 -2 1]

TEST 3: Ma trận chữ nhật
Ma trận:
[          1           0           2          -1 ]
[          2           1           4          -3 ]
Hạng (rank): 2

Cơ sở không gian cột:
  [1 2]
  [0 1]

Cơ sở không gian dòng:
  [2 1 4 -3]
  [0 -0.5 0 0.5]

Cơ sở không gian nghiệm:
  [-2 0 1 0]
  [1 1 0 1]


TEST 4: Ma trận rank = 1
Ma trận:
[          1           2           3 ]
[          2           4           6 ]
[          3           6 

## Kiểm chứng sử dụng các hàm có sẵn trong thư viện NumPy
### Hỗ trợ: GitHub Copilot (Giải thích các hàm NumPy được sử dụng trong quá trình kiểm chứng, đọc hiểu các tham số và kết quả trả về của các hàm này)

In [20]:
import numpy as np
import sympy as sp

# Kiểm chứng nghiệm Gauss với numpy.linalg.solve
print("TEST A: Kiểm chứng nghiệm Gauss với numpy.linalg.solve")
A_test = [[2, 3], [1, 5]]
b_test = [7, 11]
my_x = gaussian_eliminate(A_test, b_test)[1]
np_x = np.linalg.solve(np.array(A_test, dtype=float), np.array(b_test, dtype=float))
print("Nghiệm (code tự làm):", [float(v) for v in my_x])
print("Nghiệm (NumPy):", np_x.tolist())
flag1 = np.allclose(np.array([float(v) for v in my_x]), np_x)
compare(flag1)
print()

# Kiểm chứng định thức với numpy.linalg.det
print("TEST B: Kiểm chứng định thức")
A_det = [[1, 2, 3], [0, 1, 4], [5, 6, 0]]
my_det = float(determinant(A_det))
np_det = float(np.linalg.det(np.array(A_det, dtype=float)))
print("det (code tự làm):", my_det)
print("det (NumPy):", np_det)
flag2 = np.isclose(my_det, np_det)
compare(flag2)
print()

# Kiểm chứng nghịch đảo qua A * A^(-1) ≈ I
print("TEST C: Kiểm chứng nghịch đảo qua A * A^(-1) ≈ I")
A_inv_test = [[2, 3], [1, 5]]
my_inv = inverse(A_inv_test)
my_inv_np = np.array([[float(v) for v in row] for row in my_inv], dtype=float)
A_inv_np = np.array(A_inv_test, dtype=float)
I_calc = A_inv_np @ my_inv_np
print("A^(-1) (code tự làm) =")
print(my_inv_np)
print("A^(-1) (NumPy) =")
print(np.linalg.inv(A_inv_np))
flag3 = np.allclose(I_calc, np.eye(2))
compare(flag3)
print("A * A^(-1) =")
print(I_calc)
print()

# Kiểm chứng cơ sở không gian cột/dòng/nghiệm với SymPy
def ma_tran_co_so_cot(co_so, so_chieu):
    if not co_so:
        return np.zeros((so_chieu, 0), dtype=float)
    return np.array(co_so, dtype=float).T

# Kiểm tra xem vector v có nằm trong tập sinh của ma trận M không.
def nam_trong_tap_sinh(v, M, tol=1e-8):
    if M.shape[1] == 0:
        return np.linalg.norm(v) <= tol
    he_so, *_ = np.linalg.lstsq(M, v, rcond=None)
    return np.linalg.norm(M @ he_so - v) <= tol

# Kiểm chứng hai tập cơ sở có cùng không gian con hay không.
def cung_khong_gian_con(co_so_1, co_so_2, so_chieu, tol=1e-8):
    M1 = ma_tran_co_so_cot(co_so_1, so_chieu)
    M2 = ma_tran_co_so_cot(co_so_2, so_chieu)

    hang_1 = np.linalg.matrix_rank(M1, tol)
    hang_2 = np.linalg.matrix_rank(M2, tol)
    hang_ghep = np.linalg.matrix_rank(np.hstack([M1, M2]), tol)
    if not (hang_1 == hang_2 == hang_ghep):
        return False

    for j in range(M1.shape[1]):
        if not nam_trong_tap_sinh(M1[:, j], M2, tol):
            return False
    for j in range(M2.shape[1]):
        if not nam_trong_tap_sinh(M2[:, j], M1, tol):
            return False
    return True

# Kiểm chứng cơ sở không gian cột/dòng/nghiệm với SymPy
print("TEST D: So sánh cơ sở KG cột/dòng/nghiệm với SymPy")
A_basis = [[1, 2, 3], [2, 4, 6], [0, 1, 2]]
my_rank, my_col_space, my_row_space, my_null_space = rank_and_basis(A_basis)

A_sp = sp.Matrix(A_basis)
sp_col_space = [list(map(float, v)) for v in A_sp.columnspace()]
sp_row_space = [list(map(float, list(v))) for v in A_sp.rowspace()]
sp_null_space = [list(map(float, v)) for v in A_sp.nullspace()]

m = len(A_basis)
n = len(A_basis[0])

# Cơ sở không gian dòng có vector độ dài n, so sánh trong R^n.
my_row_as_cols = [row[:] for row in my_row_space]
sp_row_as_cols = [row[:] for row in sp_row_space]

flag_col = cung_khong_gian_con(my_col_space, sp_col_space, so_chieu=m)
flag_row = cung_khong_gian_con(my_row_as_cols, sp_row_as_cols, so_chieu=n)
flag_null = cung_khong_gian_con(my_null_space, sp_null_space, so_chieu=n)

print("Số chiều KG cột (code, SymPy):", len(my_col_space), len(sp_col_space))
print("Số chiều KG dòng (code, SymPy):", len(my_row_space), len(sp_row_space))
print("Số chiều KG nghiệm (code, SymPy):", len(my_null_space), len(sp_null_space))
print("Không gian cột cùng tập sinh:", flag_col)
print("Không gian dòng cùng tập sinh:", flag_row)
print("Không gian nghiệm cùng tập sinh:", flag_null)
compare(flag_col and flag_row and flag_null)
print()

TEST A: Kiểm chứng nghiệm Gauss với numpy.linalg.solve
Nghiệm (code tự làm): [0.2857142857142856, 2.142857142857143]
Nghiệm (NumPy): [0.2857142857142856, 2.142857142857143]
Đáp án đúng

TEST B: Kiểm chứng định thức
det (code tự làm): 0.9999999999999964
det (NumPy): 0.9999999999999964
Đáp án đúng

TEST C: Kiểm chứng nghịch đảo qua A * A^(-1) ≈ I
A^(-1) (code tự làm) =
[[ 0.71428571 -0.42857143]
 [-0.14285714  0.28571429]]
A^(-1) (NumPy) =
[[ 0.71428571 -0.42857143]
 [-0.14285714  0.28571429]]
Đáp án đúng
A * A^(-1) =
[[1.00000000e+00 0.00000000e+00]
 [5.55111512e-17 1.00000000e+00]]

TEST D: So sánh cơ sở KG cột/dòng/nghiệm với SymPy
Số chiều KG cột (code, SymPy): 2 2
Số chiều KG dòng (code, SymPy): 2 2
Số chiều KG nghiệm (code, SymPy): 1 1
Không gian cột cùng tập sinh: True
Không gian dòng cùng tập sinh: True
Không gian nghiệm cùng tập sinh: True
Đáp án đúng

